In [5]:
# Step 1: Load Dataset and Display Dimensions
import pandas as pd

# Load the raw dataset
df = pd.read_csv("/content/LungCancer_Dataset.csv")

# Print initial rows and columns
print(f"Initial Rows: {df.shape[0]}, Initial Columns: {df.shape[1]}")
print("\nFirst 5 rows:")
print(df.head())

Initial Rows: 3000, Initial Columns: 16

First 5 rows:
  GENDER  AGE  SMOKING  YELLOW_FINGERS  ANXIETY  PEER_PRESSURE  \
0      M   65        1               1        1              2   
1      F   55        1               2        2              1   
2      F   78        2               2        1              1   
3      M   60        2               1        1              1   
4      F   80        1               1        2              1   

   CHRONIC_DISEASE  FATIGUE  ALLERGY  WHEEZING  ALCOHOL_CONSUMING  COUGHING  \
0                2        1        2         2                  2         2   
1                1        2        2         2                  1         1   
2                1        2        1         2                  1         1   
3                2        1        2         1                  1         2   
4                1        2        1         2                  1         1   

   SHORTNESS_OF_BREATH  SWALLOWING_DIFFICULTY  CHEST_PAIN LUNG_CANCER  
0

In [6]:
# Step 2: Remove Duplicates
print(f"Duplicate rows found: {df.duplicated().sum()}")

# Drop duplicate rows across all columns
df = df.drop_duplicates().reset_index(drop=True)

print(f"Rows after removing duplicates: {df.shape[0]}")

Duplicate rows found: 2
Rows after removing duplicates: 2998


In [7]:
# Step 3: Correct Column Names and Data Types
import numpy as np

# Clean column names (convert to lowercase and replace spaces with underscores)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w\s]", "", regex=True)
)

# Example: Convert string columns to numeric where applicable (e.g., stripping '$' or commas)
# Replace 'price_col' with your actual column name if applicable
for col in df.select_dtypes(include=["object"]).columns:
    try:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        df[col] = pd.to_numeric(df[col])
    except (ValueError, TypeError):
        pass  # Keep as string if conversion is not valid

# Example: Convert date columns to datetime
# df['date_col'] = pd.to_datetime(df['date_col'], errors='coerce')

print("Updated Data Types:")
print(df.dtypes)

Updated Data Types:
gender                   object
age                       int64
smoking                   int64
yellow_fingers            int64
anxiety                   int64
peer_pressure             int64
chronic_disease           int64
fatigue                   int64
allergy                   int64
wheezing                  int64
alcohol_consuming         int64
coughing                  int64
shortness_of_breath       int64
swallowing_difficulty     int64
chest_pain                int64
lung_cancer              object
dtype: object


In [8]:
# Step 4: Handle Missing Values
print("Missing values per column before imputation:")
print(df.isnull().sum())

# Handle numerical columns: Fill missing values with Median
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Handle categorical columns: Fill missing values with Mode
cat_cols = df.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after imputation:")
print(df.isnull().sum().sum())

Missing values per column before imputation:
gender                   0
age                      0
smoking                  0
yellow_fingers           0
anxiety                  0
peer_pressure            0
chronic_disease          0
fatigue                  0
allergy                  0
wheezing                 0
alcohol_consuming        0
coughing                 0
shortness_of_breath      0
swallowing_difficulty    0
chest_pain               0
lung_cancer              0
dtype: int64

Missing values after imputation:
0


In [9]:
# Step 5: Detect and Treat Outliers
# Capping outliers using the Interquartile Range (IQR) method
num_cols = df.select_dtypes(include=[np.number]).columns

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap values outside lower and upper bounds
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])

print("Outliers capped using IQR bounds across all numerical columns.")

Outliers capped using IQR bounds across all numerical columns.


In [12]:
# Step 6: Feature Extraction (Combined)
import pandas as pd
import numpy as np

print(f"Columns before feature extraction: {df.shape[1]}")

# 1. Convert and extract from Datetime columns
for col in df.select_dtypes(include=['datetime64', 'object']).columns:
    if 'date' in col.lower() or 'time' in col.lower():
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df[f'{col}_year'] = df[col].dt.year
        df[f'{col}_month'] = df[col].dt.month
        df[f'{col}_dayofweek'] = df[col].dt.dayofweek
        df.drop(columns=[col], inplace=True)

# 2. Generate interaction features for first two numeric columns
num_cols = df.select_dtypes(include=[np.number]).columns
if len(num_cols) >= 2:
    c1, c2 = num_cols[0], num_cols[1]
    df[f'{c1}_per_{c2}'] = df[c1] / (df[c2] + 1e-5)

print(f"Columns after feature extraction: {df.shape[1]}")

Columns before feature extraction: 17
Columns after feature extraction: 18


In [13]:
# Step 7: Encode Categorical Variables
# Apply One-Hot Encoding to categorical features
cat_cols = df.select_dtypes(include=["object", "category"]).columns

if len(cat_cols) > 0:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

print("Categorical columns encoded successfully.")
print(f"Current shape after encoding: {df.shape}")

Categorical columns encoded successfully.
Current shape after encoding: (2998, 18)


In [14]:
# Step 8: Normalize and Standardize Numerical Features
from sklearn.preprocessing import StandardScaler

# Select numerical columns (excluding binary 0/1 columns from encoding)
num_cols = [
    col
    for col in df.columns
    if df[col].dtype in [np.float64, np.int64] and df[col].nunique() > 2
]

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print("Numerical columns scaled using StandardScaler.")

Numerical columns scaled using StandardScaler.


In [16]:
# Step 9: Save and Download Cleaned Dataset
# Save to CSV
output_filename = "LungCancer_cleaned_dataset.csv"
df.to_csv(output_filename, index=False)

print(f"Final dataset saved successfully as '{output_filename}'!")
print(
    f"Final Dimensions -> Rows: {df.shape[0]}, Columns: {df.shape[1]}"
)

# Optional: Code to automatically download if running in Google Colab
try:
    from google.colab import files

    files.download(output_filename)
except ImportError:
    print(
        f"File is saved locally in your current directory as '{output_filename}'."
    )

Final dataset saved successfully as 'LungCancer_cleaned_dataset.csv'!
Final Dimensions -> Rows: 2998, Columns: 18


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>